┌─────────────────────────────────────────────────────────┐
│                    DATA LAYER                           │
│  (Your existing pipeline — historical + live streaming) │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  FEATURE LAYER                          │
│  (Alignment, resampling, all engineered features)       │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  SIGNAL LAYER                           │
│  (Individual setups A/B/C/D — each independently)      │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  RISK LAYER                             │
│  (Position sizing, max drawdown, correlation limits)    │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│               EXECUTION LAYER                           │
│  (Order routing, slippage, Binance API)                 │
└─────────────────────────────────────────────────────────┘

In [ ]:
from orderflow import futures_agg_trades, get_oi, get_mark_price_klines, spot_agg_trades, get_premium_index_klines
import pandas as pd

symbol = "BTCUSDT"
date = "2026-04-27"

fut_trades = futures_agg_trades(symbol, date)
oi = get_oi(symbol, date)
context = get_mark_price_klines(symbol, date)
spot = spot_agg_trades(symbol, date)
premium = get_premium_index_klines(symbol, date)

mark_5m = context.resample('5min').agg({"open":'first',"high":"max", "low":"min", "close":"last"})

premium_5min = premium["close"].resample("5min").last()
R = 0.0001
funding_raw = premium_5min + R
funding_5min = funding_raw.clip(-0.0005, 0.0005)
funding_5min.name = 'funding_rate'
funding_5min = pd.DataFrame(funding_5min)
funding_5min["8h_avg"] = funding_5min["funding_rate"].rolling("0.5h").mean()
combined = pd.concat([fut_trades['fut_cumulative_volume_delta'], spot['spot_cumulative_volume_delta'], mark_5m, oi['sum_open_interest'], funding_5min], axis=1)

/var/folders/xf/n_m7ztrx4x577r3935n_1m1w0000gn/T/ipykernel_77290/1185823767.py:22: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  combined = pd.concat([fut_trades['fut_cumulative_volume_delta'], spot['spot_cumulative_volume_delta'], mark_5m, oi['sum_open_interest'], funding_5min], axis=1)


In [12]:
combined

,fut_cumulative_volume_delta,spot_cumulative_volume_delta,open,high,low,close,sum_open_interest,funding_rate,8h_avg
2026-03-27 00:00:00,-255.740,NaN,68788.100000,68801.600000,68697.901464,68763.823428,NaN,-0.000457,-0.000457
2026-03-27 00:05:00,11.054,NaN,68759.719080,68921.668572,68753.401899,68914.109468,87804.052,-0.000500,-0.000478
2026-03-27 00:10:00,-109.904,NaN,68913.315990,68942.100000,68867.591486,68901.993186,87736.673,-0.000415,-0.000457
2026-03-27 00:15:00,-105.404,NaN,68902.247534,68930.702087,68845.300891,68881.201268,87846.982,-0.000478,-0.000462
2026-03-27 00:20:00,-132.844,NaN,68880.968659,68888.400000,68706.200000,68771.500000,87882.698,-0.000371,-0.000444
...,...,...,...,...,...,...,...,...,...
58206-07-25 15:20:00,NaN,523.04162,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58206-07-29 02:40:00,NaN,511.18902,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58206-08-01 14:00:00,NaN,537.39111,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58206-08-05 01:20:00,NaN,567.12446,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# ── Usage ────────────────────────────────────────────────────────────────────
from orderflow import build_order_flow_chart

fig = build_order_flow_chart(combined, date="2024-01-15")
fig.show()